# Stellar Classification Dataset - SDSS17

## Context
In astronomy, stellar classification is the classification of stars based on their spectral characteristics. The classification scheme of galaxies, quasars, and stars is one of the most fundamental in astronomy. The early cataloguing of stars and their distribution in the sky has led to the understanding that they make up our own galaxy and, following the distinction that Andromeda was a separate galaxy to our own, numerous galaxies began to be surveyed as more powerful telescopes were built. This datasat aims to classificate stars, galaxies, and quasars based on their spectral characteristics (acc:98%).

## Content
The data consists of 100,000 observations of space taken by the SDSS (Sloan Digital Sky Survey). Every observation is described by 17 feature columns and 1 class column which identifies it to be either a star, galaxy or quasar.

1. class = object class (galaxy, star or quasar object)

## Citation

1. https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17
2. Abdurro’uf et al., The Seventeenth data release of the Sloan Digital Sky Surveys: Complete Release of MaNGA, MaStar and APOGEE-2 DATA (Abdurro’uf et al. submitted to ApJS) [arXiv:2112.02026]

In [1]:
import os
import numpy as np
import pandas as pd
from FL_cpp_method import analyze_dataset, plot_cpp
import utils  # 导入绘图基础依赖模块


In [2]:
# ==================================================
# 💡 补齐代码：多分类带保底机制的 Dirichlet 划分算法
# ==================================================
def dirichlet_multiclass_allocation(Y, Yhat, alpha_dir, num_clients, min_samples_per_client=100):
    """
    针对多分类离散标签数据的 Dirichlet 非独立同分布划分方案（含客户端最低数据量硬保障）。
    """
    classes = np.unique(Y)
    num_classes = len(classes)
    
    # 按类别分离全局样本索引并做初始乱序
    class_indices = {c: np.where(Y == c)[0] for c in classes}
    for c in classes:
        np.random.shuffle(class_indices[c])
        
    client_indices = [[] for _ in range(num_clients)]
    
    # 第一阶段：保底分配。确保每一个客户端都能分到基础数量的样本，彻底避免节点掏空
    base_per_class = min_samples_per_client // num_classes
    if base_per_class < 1:
        base_per_class = 1
        
    for c in classes:
        indices = class_indices[c]
        available = len(indices)
        required = base_per_class * num_clients
        actual_base = base_per_class if required <= available else available // num_clients
        
        if actual_base > 0:
            for i in range(num_clients):
                start = i * actual_base
                end = (i + 1) * actual_base
                client_indices[i].extend(indices[start:end])
            class_indices[c] = indices[num_clients * actual_base:]
            
    # 第二阶段：迪利克雷异质偏斜分配。剩余的自由数据严格按照 Dirichlet 的采样矩阵分发
    for c in classes:
        indices = class_indices[c]
        if len(indices) == 0:
            continue
            
        proportions = np.random.dirichlet([alpha_dir] * num_clients)
        counts = np.floor(proportions * len(indices)).astype(int)
        
        # 将向下取整带来的多余残差随机指派给节点
        remainder = len(indices) - np.sum(counts)
        for _ in range(remainder):
            idx = np.random.choice(num_clients)
            counts[idx] += 1
            
        start = 0
        for i in range(num_clients):
            end = start + counts[i]
            client_indices[i].extend(indices[start:end])
            start = end
            
    # 拼合重排最终的全局索引数组
    reordered_indices = []
    actual_sizes = []
    for i in range(num_clients):
        np.random.shuffle(client_indices[i])
        reordered_indices.extend(client_indices[i])
        actual_sizes.append(len(client_indices[i]))
        
    reordered_indices = np.array(reordered_indices, dtype=int)
    return Y[reordered_indices], Yhat[reordered_indices], actual_sizes


In [3]:
# %%
# ==================================================
# 基本实验控制元参数
# ==================================================
dataset_name = 'stellar'
alpha = 0.1
method = "mean"
num_clients = 20  
xlim = [0, 2]
ylim = [0, 1.0]

# 核心暂存队列：用于跨准确率合并数据并输出多合一总表
iid_total_records = []
non_iid_total_records = []
acc_steps = np.arange(0.1, 1.1, 0.1)

# ==================================================
# 纵向大循环核心
# ==================================================
for acc in acc_steps:
    acc_str = f"{acc:.1f}"
    data_path = f'../data/stellar/stellar_acc_{acc_str}.npz'
    
    if not os.path.exists(data_path):
        print(f"⚠️ [跳过] 未检测到文件: {data_path}")
        continue
        
    # 🛠️ 更正控制台日志符号
    print(f"\n⚡ [当前进度] 正在全面计算精度级别 -> Acc = {acc_str}")
    data = np.load(data_path)
    Y_total = data["Y"]
    Yhat_total = data["Y_hat"]
    
    # --------------------------------------------------
    # 1. 运行 IID 实验并画图
    # --------------------------------------------------
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, None, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, grid=None
    )
    
    # 🛠️ 更正 X 轴标签文本：λ 纠正为 Acc
    title_iid = f"Acc = {acc_str} (IID)"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, acc_str, None, xlim, ylim, title_iid, None)
    
    iid_total_records.append({
        'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # --------------------------------------------------
    # 2. 运行 三种 Dirichlet Non-IID 实验并画图
    # --------------------------------------------------
    alpha_dir_list = [1.0, 0.1, 0.01]
    dataset_dist_non = 'Non-IID'
    
    for alpha_dir in alpha_dir_list:
        Y_dir, Yhat_dir, actual_sizes = dirichlet_multiclass_allocation(
            Y_total, Yhat_total, alpha_dir, num_clients, min_samples_per_client=100
        )
        
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, None, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid=None
        )
        
        # 🛠️ 更正 X 轴标签文本：移除 Dirichlet 字段，对齐 \alpha_{Dir} 和 Acc 符号
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$ (Acc = {acc_str})"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, acc_str, alpha_dir, xlim, ylim, title_dir, None)
        
        non_iid_total_records.append({
            'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# ==================================================
# 3. 多合一历史指标总表落盘归档
# ==================================================
csv_flat_dir = os.path.join('.', 'result', dataset_name, 'csv')
os.makedirs(csv_flat_dir, exist_ok=True)

if len(iid_total_records) > 0:
    iid_master_path = os.path.join(csv_flat_dir, f'{dataset_name}_IID_summary.csv')
    pd.DataFrame(iid_total_records).to_csv(iid_master_path, index=False)
    print(f"\n📊 [Master CSV 成功] IID 历史总表汇总至: {iid_master_path}")

if len(non_iid_total_records) > 0:
    non_master_path = os.path.join(csv_flat_dir, f'{dataset_name}_Non-IID_summary.csv')
    pd.DataFrame(non_iid_total_records).to_csv(non_master_path, index=False)
    print(f"📊 [Master CSV 成功] Non-IID 历史总表汇总至: {non_master_path}")


⚡ [当前进度] 正在全面计算精度级别 -> Acc = 0.1
labeled_ratio 0.3
分组： 1
带标签的样本量： 2281
不带标签的样本量： 5324
分组： 2
带标签的样本量： 2281
不带标签的样本量： 5324
分组： 3
带标签的样本量： 2281
不带标签的样本量： 5324
分组： 4
带标签的样本量： 2281
不带标签的样本量： 5324
分组： 5
带标签的样本量： 2281
不带标签的样本量： 5324
分组： 6
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 7
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 8
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 9
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 10
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 11
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 12
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 13
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 14
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 15
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 16
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 17
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 18
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 19
带标签的样本量： 2281
不带标签的样本量： 5323
分组： 20
带标签的样本量： 2281
不带标签的样本量： 5323
带标签的样本量： 45620
不带标签的样本量： 106465

最终结果：
真实 theta: 1.0
CPP intervals: [array([0.953798  , 1.01025269]), array([0.98959555, 1.04547693]), array([0.99044136, 1.04726197]), array([0.92047261, 0.97694085]), array([0.94093724, 0.99680913]), arr

In [4]:
# %%
# ==================================================
# 1. 固定准确率为 50% 的核心数据集加载与基准参数配置
# ==================================================
dataset_name = 'stellar'
fixed_acc_str = '0.5'  
data_path = f'../data/stellar/stellar_acc_{fixed_acc_str}.npz'

print(f"正在载入固定 50% 准确率目标数据包: {data_path}")
data = np.load(data_path)
Y_total = data["Y"]
Yhat_total = data["Y_hat"]

alpha = 0.1  
method = "mean"
num_clients = 20  
xlim = [0, 2]
ylim = [0, 1.0]

iid_ratio_records = []
non_iid_ratio_records = []
ratio_steps = [0.1, 0.2, 0.3, 0.4, 0.5]

# ==================================================
# 2. 纵向多标签比例阶梯对比大循环核心
# ==================================================
for ratio in ratio_steps:
    ratio_str = f"{ratio:.1f}"
    sub_folder_name = f"ratio_{ratio_str}"  
    print(f"\n🚀 [实验进行中] 正在注入有标签数据点比例阶梯 -> labeled_ratio = {ratio_str}")
    
    # --------------------------------------------------
    # 2.1 当前 ratio 级别下的标准 IID 实验
    # --------------------------------------------------
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, None, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, None, current_ratio=ratio
    )
    
    # 🛠️ 更正 X 轴标签文本：用 Acc 表示精度，用 \lambda 表示少样本抽样比
    title_iid = f"Acc = 0.5 | $\\lambda = {ratio_str}$ (IID)"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, fixed_acc_str, None, xlim, ylim, title_iid, sub_folder_name)
    
    iid_ratio_records.append({
        'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # --------------------------------------------------
    # 2.2 当前 ratio 级别下的三种 Dirichlet Non-IID 循环
    # --------------------------------------------------
    alpha_dir_list = [1.0, 0.1, 0.01]
    dataset_dist_non = 'Non-IID'
    
    for alpha_dir in alpha_dir_list:
        Y_dir, Yhat_dir, actual_sizes = dirichlet_multiclass_allocation(
            Y_total, Yhat_total, alpha_dir, num_clients, min_samples_per_client=100
        )
        
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, None, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid=None, current_ratio=ratio
        )
        
        # 🛠️ 更正 X 轴标签文本：完全剔除 Dirichlet 单词，精准使用 \alpha_{Dir} 与 \lambda
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$ ($\\lambda = {ratio_str}$)"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, fixed_acc_str, alpha_dir, xlim, ylim, title_dir, sub_folder_name)
        
        non_iid_ratio_records.append({
            'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# ==================================================
# 3. 数据总结合并归档
# ==================================================
csv_master_dir = os.path.join('.', 'result', dataset_name, 'csv')
os.makedirs(csv_master_dir, exist_ok=True)

if len(iid_ratio_records) > 0:
    iid_ratio_path = os.path.join(csv_master_dir, f'{dataset_name}_IID_ratio_summary.csv')
    pd.DataFrame(iid_ratio_records).to_csv(iid_ratio_path, index=False)
    print(f"\n📊 [Master CSV 成功] IID 全标签比例阶梯总表已汇流至: {iid_ratio_path}")

if len(non_iid_ratio_records) > 0:
    non_ratio_path = os.path.join(csv_master_dir, f'{dataset_name}_Non-IID_ratio_summary.csv')
    pd.DataFrame(non_iid_ratio_records).to_csv(non_ratio_path, index=False)
    print(f"📊 [Master CSV 成功] Non-IID 全标签比例与异质度交叉总表已汇流至: {non_ratio_path}")

正在载入固定 50% 准确率目标数据包: ../data/stellar/stellar_acc_0.5.npz

🚀 [实验进行中] 正在注入有标签数据点比例阶梯 -> labeled_ratio = 0.1
labeled_ratio 0.1
分组： 1
带标签的样本量： 760
不带标签的样本量： 6845
分组： 2
带标签的样本量： 760
不带标签的样本量： 6845
分组： 3
带标签的样本量： 760
不带标签的样本量： 6845
分组： 4
带标签的样本量： 760
不带标签的样本量： 6845
分组： 5
带标签的样本量： 760
不带标签的样本量： 6845
分组： 6
带标签的样本量： 760
不带标签的样本量： 6844
分组： 7
带标签的样本量： 760
不带标签的样本量： 6844
分组： 8
带标签的样本量： 760
不带标签的样本量： 6844
分组： 9
带标签的样本量： 760
不带标签的样本量： 6844
分组： 10
带标签的样本量： 760
不带标签的样本量： 6844
分组： 11
带标签的样本量： 760
不带标签的样本量： 6844
分组： 12
带标签的样本量： 760
不带标签的样本量： 6844
分组： 13
带标签的样本量： 760
不带标签的样本量： 6844
分组： 14
带标签的样本量： 760
不带标签的样本量： 6844
分组： 15
带标签的样本量： 760
不带标签的样本量： 6844
分组： 16
带标签的样本量： 760
不带标签的样本量： 6844
分组： 17
带标签的样本量： 760
不带标签的样本量： 6844
分组： 18
带标签的样本量： 760
不带标签的样本量： 6844
分组： 19
带标签的样本量： 760
不带标签的样本量： 6844
分组： 20
带标签的样本量： 760
不带标签的样本量： 6844
带标签的样本量： 15200
不带标签的样本量： 136885

最终结果：
真实 theta: 1.0
CPP intervals: [array([0.93058144, 1.02568843]), array([0.97077008, 1.06447919]), array([0.96660228, 1.06143882]), array([0.89988543